# 第 1 周 Day 4：Causal Mask 与 Multi-Head Attention — Notebook 作业

[← Week 01 / Day 03](day-03.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-04.md) · [Goal 进度](../../PROGRESS.md) · [Week 01 / Day 05 →](day-05.ipynb)

> 状态：**未提交**。直接编辑各个 Markdown/Code 单元格；“教练验收区”不要预填。


## Goal

预计 90–120 分钟。能构造 causal mask，解释为什么自回归模型不能读取未来 token；能推导多头注意力从 `[B,N,D]` 到 `[B,H,N,D_h]` 再合并的形状。

### 我的目标复述

【双击此 Markdown 单元格，用自己的话填写今日目标及其在 VLA 中的作用。】


## Setup

| 字段 | 我的记录 |
|---|---|
| 实际投入时间 | 【填写】 |
| 完成日期 | 【填写】 |
| Python / PyTorch | 【填写；纯理论日写“不适用”】 |
| CPU / GPU / 仿真器 | 【填写】 |
| 资源等级 | 【L0 / L1 / L2】 |
| 产物路径 | 【填写】 |

生成时环境检查（2026-09-01）：当前可见 Python 未检测到 Jupyter、ipykernel、nbformat、PyTorch 或 NumPy。本 Notebook 已做结构验证，但在该环境中尚未执行。


In [ ]:
# 可选：Notebook 环境可用后运行此单元，记录基础环境。
import platform
import sys

print("python:", sys.version)
print("platform:", platform.platform())


## Context：知识点及其在 VLA 中的作用

Causal mask 约束第 `i` 个位置只能读取不晚于 `i` 的位置，避免训练时偷看未来动作或文本。Multi-head attention 将特征拆成多个子空间，使不同头可学习对象、空间、动作阶段等不同匹配关系；“可学习”不等于每个头必然具有可读语义。


## Concepts：概念、公式、形状与数据流

长度 `N=4` 的允许矩阵为下三角：

```text
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```

实现时常将禁止位置在 softmax 前填为 `-inf`。设 `D=32,H=4,D_h=8`：

```text
X              [B,N,32]
Q/K/V 投影      [B,N,32]
split heads    [B,4,N,8]
attention      [B,4,N,N]
head outputs   [B,4,N,8]
concat         [B,N,32]
```

`D` 必须能被头数 `H` 整除。padding mask 与 causal mask 目的不同：前者屏蔽补齐位置，后者屏蔽未来位置。


## Learning Steps

1. 画 `4×4` mask，逐行解释可见范围。
2. 将 mask 加到 Day 3 的 scores 上，再做 softmax。
3. 用 `B=2,N=5,D=32,H=4` 手推每一步形状。
4. 用 `torch.nn.MultiheadAttention(embed_dim=32,num_heads=4,batch_first=True)` 跑最小样例。
5. 比较无 mask 与 causal mask 下第 1 个位置的输出。


## Steps：必做作业

### 课程题目

实现 `make_causal_mask(n)` 并生成 `n=4` mask；把它应用到单头 attention，验证所有未来权重接近 0。再运行多头模块，输入 `[2,5,32]`，打印输出和权重形状。若框架默认返回跨头平均权重，需在记录中说明，并尝试设置不平均的选项或只提交文档推导。

下面每道题都有独立作答单元。文字、表格、公式或 Mermaid 写在 Markdown 单元；可运行代码写在后面的 Code 单元。


### 第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 可运行代码 / 实验区

纯理论日可以保留为空；代码日请将实现拆成短小单元，并保留关键输出。


In [ ]:
# 在此编写或运行当天代码。
# 建议先写清输入 shape、dtype、设备和随机种子。


## Checks：输入、预期输出与验证

- 输入：随机 `x [2,5,32]`，4 heads；`n=4` 的因果测试张量。
- 预期：MHA 输出 `[2,5,32]`；未平均权重通常为 `[2,4,5,5]`；未来位置权重小于 `1e-6`。
- 验证：检查上三角（不含对角）权重最大值；确认每头 `D_h=8`；输出均为有限数。

低资源替代：完整手画形状图与 `4×4` masked scores，解释每行 softmax 范围。

### 我的验证记录

| 检查项 | 实际结果 | 是否符合 | 证据 |
|---|---|---|---|
| 输入 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| 输出 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| dtype、范围和单位 | 【填写】 | 【填写】 | 【填写】 |
| 正向测试 | 【填写】 | 【填写】 | 【填写】 |
| 负向测试 / 错误注入 | 【填写】 | 【填写】 | 【填写】 |
| 指标分子 / 分母 / seed | 【填写】 | 【填写】 | 【填写】 |

> 尚未运行的内容必须标为“预期结果”，不能作为实际证据。


In [ ]:
# 在此编写 shape、dtype、数值范围、断言或负向测试。


## Evidence：提交与复现证据

固定包含：`causal mask 图`、`多头形状链`、`代码或手算`、`未来权重验证`、`框架权重返回说明`、`投入分钟数`。

### 我的证据

- 代码路径：【填写】
- 配置路径：【填写】
- 数据 / checkpoint / commit 或哈希：【填写】
- 实际命令：【填写】
- 退出码：【填写】
- 关键输出：【填写】
- 结果说明了什么：【填写】
- 结果没有说明什么：【填写】
- 失败现象与定位证据：【填写】

### VLA 约束

| 约束 | 我的定义 |
|---|---|
| 图像布局、颜色顺序和范围 | 【填写】 |
| 文本 token、padding 与 mask | 【填写】 |
| 机器人状态各维含义 | 【填写】 |
| 坐标系、长度和角度单位 | 【填写】 |
| 动作空间及逐维定义 | 【填写】 |
| observation/action 时间对齐 | 【填写】 |
| 控制频率 / action chunk | 【填写】 |
| 归一化及统计量来源 | 【填写】 |
| 随机种子与数据划分 | 【填写】 |


## Self-check：课程自测

1. 视觉编码器是否总需要 causal mask？
2. 动作序列自回归训练为何需要它？
3. `D=30,H=8` 能直接均分吗？
4. 多头权重被平均后会丢失什么信息？


### 自测第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 4 题作答

【双击此 Markdown 单元格，在这里填写答案。】


## Help：排查、最低完成线与提高

### 常见错误

- mask 方向反了：第 0 行只能看第 0 列。
- softmax 后才置零：权重不再归一，应在 softmax 前屏蔽。
- 全行都是 `-inf` 导致 NaN：至少允许当前位置或有效历史。
- 忽略框架的 `batch_first`：检查输入文档与输出第一维。
- 将平均后的 `[B,N,N]` 误报为每头权重。

### 最低完成线

画对 `4×4` causal mask，推导 `B=2,N=5,D=32,H=4` 全部形状，并解释两个 mask 的区别。

### 可选提高

同时组合 padding mask 与 causal mask，对长度分别为 3 和 5 的两个序列检查有效权重区域。


## Rubric

100 分，80 分通过：mask 语义 20；mask 实现与验证 25；多头形状 25；框架实验 15；padding/causal 区分 15。若允许未来信息或错误地在 softmax 后加 mask，关键项失败。

### 提交前检查

- [ ] 已逐项完成必做作业；
- [ ] 已区分实际结果与预期结果；
- [ ] 已保留代码输出、日志、表格或推理证据；
- [ ] 已记录适用的 shape、坐标系、单位、动作和时间约定；
- [ ] 已完成验证或明确写出无法执行的原因；
- [ ] 已回答全部自测题；
- [ ] 已记录仍不确定的点或失败案例。


## Coach Review（学习者请勿填写）

| 字段 | 验收结果 |
|---|---|
| 证据完整性 | 待验收 |
| Rubric 得分 | /100 |
| 门槛项 | 待验收 |
| 当天状态 | 未提交 |
| 具体缺口 |  |
| 最小补救任务 |  |
| 复验结果 |  |
| 下一课程 |  |


## Next Steps

完成后保存 Notebook，并把路径发到学习对话：

`docs/vla-learning/notebooks/week-01/day-04.ipynb`

教练验收通过后才会更新 `PROGRESS.md`。

[← Week 01 / Day 03](day-03.ipynb) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-04.md) · [Week 01 / Day 05 →](day-05.ipynb)
